# layers

> Building blocks the MAX graph needs that are not in `max.experimental.nn`.
> 
> Activations are channels last, `(batch, time, channels)`, all the way
> through. A 1D convolution is a 2D convolution on a height-one image.


In [ ]:
#| default_exp layers


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
from max.dtype import DType
from max.experimental import functional as F
from max.experimental.nn import Linear, Module
from max.experimental.tensor import Tensor
from max.graph.type import FilterLayout


`Conv1d` keeps its weight in the PyTorch layout, `(out, in / groups, kernel)`, so checkpoint keys and shapes load as they are. The graph rearranges it into the `(1, kernel, in / groups, out)` filter that `conv2d` takes. This conv runs on CPU, and also on GPUs without a cuDNN-only filter layout.


In [ ]:
#| export
class Conv1d(Module):
    """1D convolution over `(batch, time, channels)` with stride, padding and groups."""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        *,
        stride: int = 1,
        padding: int = 0,
        groups: int = 1,
        bias: bool = False,
    ) -> None:
        if in_channels % groups or out_channels % groups:
            raise ValueError(f"{in_channels} and {out_channels} channels must both divide into {groups} groups")
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.groups = groups
        self.weight = Tensor.zeros([out_channels, in_channels // groups, kernel_size], dtype=DType.float32)
        self.bias = Tensor.zeros([out_channels], dtype=DType.float32) if bias else None

    def forward(self, x: Tensor) -> Tensor:
        batch, steps = x.shape[0], x.shape[1]
        image = x.reshape([batch, 1, steps, self.in_channels])
        filters = self.weight.permute([2, 1, 0]).reshape(
            [1, self.kernel_size, self.in_channels // self.groups, self.out_channels]
        )
        out = F.conv2d(
            image,
            filters,
            (1, self.stride),
            (1, 1),
            (0, 0, self.padding, self.padding),
            self.groups,
            self.bias,
            filter_layout=FilterLayout.RSCF,
        )
        return out.reshape([batch, out.shape[2], self.out_channels])


The first feature-encoder layer normalizes with `GroupNorm(512, 512)`: one channel per group. That is a per-channel mean and variance over time, with a learned scale and shift. Doing it channels last avoids transposing the whole waveform.


In [ ]:
#| export
class ChannelNorm(Module):
    """`GroupNorm` with one channel per group, for `(batch, time, channels)`."""

    def __init__(self, channels: int, eps: float = 1e-5) -> None:
        self.eps = eps
        self.weight = Tensor.ones([channels], dtype=DType.float32)
        self.bias = Tensor.zeros([channels], dtype=DType.float32)

    def forward(self, x: Tensor) -> Tensor:
        centered = x - F.mean(x, axis=1)
        variance = F.mean(centered * centered, axis=1)
        return centered * F.rsqrt(variance + self.eps) * self.weight + self.bias


In [ ]:
#| export
class Wav2Vec2Attention(Module):
    """Bidirectional multi-head attention. There is no mask: base clips are not padded inside the graph."""

    def __init__(self, hidden_size: int, num_heads: int) -> None:
        if hidden_size % num_heads:
            raise ValueError(f"{hidden_size} is not divisible by {num_heads} heads")
        self.heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.scale = self.head_dim ** -0.5
        self.k_proj = Linear(hidden_size, hidden_size)
        self.v_proj = Linear(hidden_size, hidden_size)
        self.q_proj = Linear(hidden_size, hidden_size)
        self.out_proj = Linear(hidden_size, hidden_size)

    def forward(self, hidden: Tensor) -> Tensor:
        batch, seq = hidden.shape[0], hidden.shape[1]
        query = self._heads(self.q_proj(hidden), batch, seq)
        key = self._heads(self.k_proj(hidden), batch, seq)
        value = self._heads(self.v_proj(hidden), batch, seq)
        scores = (query @ key.permute([0, 1, 3, 2])) * self.scale
        context = F.softmax(scores, axis=-1) @ value
        context = context.permute([0, 2, 1, 3]).reshape([batch, seq, self.heads * self.head_dim])
        return self.out_proj(context)

    def _heads(self, states: Tensor, batch, seq) -> Tensor:
        return states.reshape([batch, seq, self.heads, self.head_dim]).permute([0, 2, 1, 3])


In [ ]:
#| export
class Wav2Vec2FeedForward(Module):
    """Linear, GELU, linear."""

    def __init__(self, hidden_size: int, intermediate_size: int) -> None:
        self.intermediate_dense = Linear(hidden_size, intermediate_size)
        self.output_dense = Linear(intermediate_size, hidden_size)

    def forward(self, hidden: Tensor) -> Tensor:
        return self.output_dense(F.gelu(self.intermediate_dense(hidden)))
